# Notebook 08 — Deploy and operate Genie

**What you’ll try:**

- **Playground** — fast demos with your Genie space.
- **Databricks App** — a small sample app you can deploy from workspace files.
- **Jobs** — refresh data on a schedule; Genie keeps reading live tables.
- **CI/CD (optional)** — export, transform, and copy a space definition across environments.

**Good habits:** Keep **`GENIE_SPACE_ID`** in app env vars or secrets—not in Git. Use **different spaces** for dev / test / prod. **Compute:** **Serverless** for this notebook.


## Compute

Use **Serverless** for this notebook. Apps and Jobs use the compute you pick in those products.


## Workspace links and Genie space id

Run this cell **before** Options A–D. It prints your **space id** and **clickable links** (Genie room, Monitoring, Playground, Jobs, Apps).

**Azure workspaces** add **`?o=<workspace_id>`** to many URLs. If a link does not open, try the **hash-route** alternate in the list, or open **Playground** from the sidebar under **Machine learning**.


In [ ]:
import html
import re

from databricks.sdk import WorkspaceClient

dbutils.widgets.text("catalog", "workshop_demo", "Catalog")
dbutils.widgets.text("schema", "genie_workshop_manufacturing", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

w = WorkspaceClient()
_raw_host = (w.config.host or "").strip().rstrip("/")


def ensure_https(url: str) -> str:
    u = (url or "").strip().rstrip("/")
    if not u:
        return u
    if not u.lower().startswith("http"):
        u = "https://" + u.lstrip("/")
    return u


host_url = ensure_https(_raw_host)


def _azure_workspace_o(host: str) -> str:
    m = re.search(r"adb-(\d+)\.", host or "")
    return m.group(1) if m else ""


def workspace_deeplink(host: str, path: str) -> str:
    # path must start with / e.g. /ml/playground
    h = ensure_https(host).rstrip("/")
    p = path if path.startswith("/") else "/" + path
    base = h + p
    oid = _azure_workspace_o(h)
    if not oid:
        return base
    sep = "&" if "?" in base else "?"
    return f"{base}{sep}o={oid}"


def genie_ui_room_url(host: str, space_id: str, *, monitoring: bool = False) -> str:
    h = ensure_https(host).rstrip("/")
    sid = str(space_id or "").strip()
    mid = "/monitoring" if monitoring else ""
    return workspace_deeplink(h, f"/genie/rooms/{sid}{mid}")


cfg = spark.sql(
    f"SELECT value FROM {CATALOG}.{SCHEMA}.workshop_config WHERE key = 'genie_space_id'"
).collect()
if not cfg:
    raise RuntimeError("Missing genie_space_id — run notebook 02.")
GENIE_SPACE_ID = cfg[0]["value"]
UI_URL = genie_ui_room_url(host_url, GENIE_SPACE_ID, monitoring=False)
MONITORING_UI_URL = genie_ui_room_url(host_url, GENIE_SPACE_ID, monitoring=True)

pg_ai = workspace_deeplink(host_url, "/ml/ai-playground")
pg_ml = workspace_deeplink(host_url, "/ml/playground")
_oid = _azure_workspace_o(host_url)
pg_hash = (
    f"{host_url}/?o={_oid}#/ml/playground"
    if _oid
    else f"{host_url}/#/ml/playground"
)
jobs_url = workspace_deeplink(host_url, "/jobs")
jobs_hash = (
    f"{host_url}/?o={_oid}#/job/list" if _oid else f"{host_url}/#/job/list"
)
apps_url = workspace_deeplink(host_url, "/compute/apps")
apps_hash = (
    f"{host_url}/?o={_oid}#/compute/apps" if _oid else f"{host_url}/#/compute/apps"
)

print("GENIE_SPACE_ID:", GENIE_SPACE_ID)
print("Genie room URL:", UI_URL)
print("Genie Monitoring deep link:", MONITORING_UI_URL)


def _a(href: str, label: str) -> str:
    return (
        '<li style="margin:6px 0"><a href="'
        + html.escape(href, quote=True)
        + '" target="_blank" rel="noopener noreferrer">'
        + html.escape(label)
        + "</a></li>"
    )


_id_show = html.escape(str(GENIE_SPACE_ID))
_html = (
    '<div style="font-family:system-ui,Segoe UI,sans-serif;line-height:1.5;max-width:720px">'
    "<p><b>Clickable workspace links</b> — open in a new tab. If a route 404s, try the <b>hash-route</b> line for your tenant.</p>"
    '<ul style="padding-left:1.2rem">'
    + _a(UI_URL, "Genie — SQL space (main room)")
    + _a(MONITORING_UI_URL, "Genie — Monitoring (direct URL)")
    + _a(pg_ai, "AI Playground — /ml/ai-playground")
    + _a(pg_ml, "AI Playground — /ml/playground (legacy path)")
    + _a(pg_hash, "AI Playground — hash route (?o=…#/ml/playground)")
    + _a(jobs_url, "Jobs — /jobs")
    + _a(jobs_hash, "Jobs — hash route (#/job/list)")
    + _a(apps_url, "Apps — /compute/apps")
    + _a(apps_hash, "Apps — hash route (#/compute/apps)")
    + "</ul>"
    '<p style="margin-top:12px;color:#555;font-size:13px">'
    "Copy space ID for Playground <b>Genie</b> tool:</p>"
    '<p style="font-family:monospace;font-size:14px;background:#f5f5f5;padding:8px;border-radius:6px">'
    + _id_show
    + "</p></div>"
)
displayHTML(_html)


## Option A — Playground / Agent Builder

1. Open **AI Playground** using a **clickable link** above (try **ai-playground** first).
2. Add **Tool** → **Genie** → paste **`GENIE_SPACE_ID`** from the gray box.
3. Ask: *What is average OEE by plant for 2024?*
4. When stable, export or wire an App as needed.


In [ ]:
print("Sample Playground question: What is average OEE by plant for 2024?")


## Option B — Databricks App (`app/`)

This workshop ships a small **Gradio** app under `app/` (Genie Conversation API). The next cell **copies** `app.py`, `app.yaml`, and `requirements.txt` into a sibling folder **`manufacturing_genie_app/`** in the same workspace directory as this notebook and injects the current **`GENIE_SPACE_ID`** into `app.yaml`.

Then open **Apps** from the links cell → **Create app** → choose **Source: Workspace** → select **`…/manufacturing_genie_app`**. After deploy, grant the app’s **service principal** `USE`/`SELECT` on your catalog/schema (and Genie **CAN USE** on the space) if you see permission errors.

Prerequisite: sync this repository’s `app/` folder next to your notebooks in the workspace (the maintainer E2E sync script can upload it). Run the **Workspace links** cell first so **`GENIE_SPACE_ID`** is defined.


In [ ]:
# Materialize the App bundle next to this notebook (workspace files).
import base64
import html
import re
from pathlib import Path

from databricks.sdk.service.workspace import ExportFormat, ImportFormat

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_np = ctx.notebookPath().get()
nb_path = _np if str(_np).startswith("/") else "/" + str(_np)
workshop_dir = str(Path(nb_path).parent)
APP_SRC = f"{workshop_dir}/app"
APP_DEPLOY = f"{workshop_dir}/manufacturing_genie_app"

w.workspace.mkdirs(APP_DEPLOY)


def _export_bytes(rel: str) -> bytes:
    # CLI sync imports app.py as workspace path .../app/app (no .py suffix).
    candidates = [f"{APP_SRC}/{rel}"]
    if rel.endswith(".py"):
        candidates.append(f"{APP_SRC}/{rel[:-3]}")
    last = None
    for path in candidates:
        try:
            er = w.workspace.export(path, format=ExportFormat.AUTO)
            return base64.b64decode(er.content)
        except Exception as ex:
            last = ex
    raise last


written = []
for name in ("app.py", "requirements.txt", "app.yaml"):
    try:
        raw = _export_bytes(name)
        if name == "app.yaml":
            txt = raw.decode("utf-8")
            if 'value: ""' in txt:
                txt = txt.replace('value: ""', f'value: "{GENIE_SPACE_ID}"', 1)
            else:
                txt = re.sub(
                    r'^(\s*value:\s*)"[^"]*"',
                    rf'\1"{GENIE_SPACE_ID}"',
                    txt,
                    count=1,
                    flags=re.MULTILINE,
                )
            raw = txt.encode("utf-8")
        w.workspace.upload(
            f"{APP_DEPLOY}/{name}",
            content=raw,
            format=ImportFormat.AUTO,
            overwrite=True,
        )
        written.append(name)
    except Exception as ex:
        print(f"Could not copy {name} from {APP_SRC}/: {ex}")

print(f"Deploy folder: {APP_DEPLOY}")
print("Files written:", written if written else "(none)")
if len(written) < 3:
    print(
        "Sync the repo folder app/ into " + APP_SRC + " (see README / E2E sync), then re-run this cell."
    )

displayHTML(
    "<p><b>Next:</b> Apps → Create app → source = Workspace → folder</p>"
    f'<p style="font-family:monospace;font-size:13px">{html.escape(APP_DEPLOY)}</p>'
)


## Option C — Job + Genie

Use **Jobs** to refresh **data** on a schedule; Genie spaces keep pointing at the same Unity Catalog tables and see new data automatically.

1. Open **Jobs** (path or hash link from the workspace links cell).
2. **Schedule notebook 01** (or your production bronze/silver jobs) with the same **`catalog`** and **`schema`** parameters you use interactively.
3. **Regression checks:** after instruction or foundation-model changes, schedule **03** (eval setup) and **06** (compare / `genie_benchmark_runs`) and alert if pass rate drops.
4. Your repo includes a **reference** multi-task definition: `scripts/job_submit_manufacturing_genie_e2e_01_09.json` (nine serverless notebook tasks) — clone tasks you need rather than running the full chain in production.


In [ ]:
import html

try:
    _ju, _jh = jobs_url, jobs_hash
except NameError as _e:
    raise RuntimeError("Run the workspace links cell above first.") from _e

job_params_example = {
    "catalog": CATALOG,
    "schema": SCHEMA,
    "notebooks_scheduled": ["01_setup_data", "03_genie_evals_benchmarks", "06_compare_genie_spaces"],
}
print("Example base_parameters for scheduled notebook tasks:", job_params_example)

_optc = (
    '<div style="font-family:system-ui,sans-serif;max-width:720px;line-height:1.5">'
    "<p><b>Open Jobs</b> and create or clone a job:</p><ul>"
    f'<li><a href="{html.escape(_ju, quote=True)}" target="_blank" rel="noopener">Jobs (/jobs)</a></li>'
    f'<li><a href="{html.escape(_jh, quote=True)}" target="_blank" rel="noopener">Jobs (hash route)</a></li>'
    "</ul>"
    "<p>Use the same <code>catalog</code> / <code>schema</code> as widgets for every scheduled task. "
    "Reference bundle: <code>scripts/job_submit_manufacturing_genie_e2e_01_09.json</code>.</p></div>"
)
displayHTML(_optc)




## Option D — CI/CD for Genie Spaces

### Why CI/CD matters

In production, you will have **separate Genie spaces per environment** (dev, staging, prod). As you iterate on instructions, curated SQL, and table configurations in dev, you need a **repeatable, auditable** way to promote those changes.

### Current state (April 2026)

| Mechanism | Status | Notes |
|-----------|--------|-------|
| **Genie CRUD REST APIs** | **Public Preview** | `POST`, `GET`, `PATCH` on `/api/2.0/genie/spaces` — the supported CI/CD path today |
| **Databricks Asset Bundles (DABs)** | In progress (DIRECT mode PR) | `databricks/cli` PR #4191 adds `genie_spaces` resource — not yet GA |
| **Terraform provider** | Not yet available | No Genie resource exposed; on roadmap after DABs |

### The 3-phase promotion pattern

```
┌──────────┐     ┌─────────────┐     ┌──────────┐
│  EXPORT  │ ──> │  TRANSFORM  │ ──> │  DEPLOY  │
│  (dev)   │     │  (CI / Git) │     │  (prod)  │
└──────────┘     └─────────────┘     └──────────┘

Phase 1: GET /api/2.0/genie/spaces/{id}?include_serialized_space=true
Phase 2: Remap catalog.schema references, commit JSON to Git
Phase 3: POST (create) or PATCH (update) to target workspace
```

**Key gotcha:** Table identifiers inside `serialized_space` are hardcoded as `catalog.schema.table`. When promoting across environments with different catalogs, you **must** remap these references.

### Phase 1 — Export the Genie space definition

Fetch the full space configuration including `serialized_space`. This captures everything: tables, instructions, curated SQL, sample questions, and benchmarks.

In [ ]:
import json
import requests
import copy

headers = {**w.config.authenticate(), "Content-Type": "application/json"}

# Phase 1: Export the full Genie space definition
export_resp = requests.get(
    f"{host_url}/api/2.0/genie/spaces/{GENIE_SPACE_ID}?include_serialized_space=true",
    headers=headers,
)
export_resp.raise_for_status()
exported = export_resp.json()

# serialized_space may be a JSON string or dict; normalize to dict
_ss = exported.get("serialized_space")
space_def = json.loads(_ss) if isinstance(_ss, str) else (_ss if isinstance(_ss, dict) else {})

print(f"Exported space: {exported.get('title') or exported.get('display_name') or 'N/A'}")
print(f"Tables: {len(space_def.get('data_sources', {}).get('tables', []))}")
print(f"Curated SQL examples: {len(space_def.get('instructions', {}).get('example_question_sqls', []))}")
print(f"Sample questions: {len(space_def.get('config', {}).get('sample_questions', []))}")
print(f"\nFull serialized_space (first 500 chars):\n{json.dumps(space_def, indent=2)[:500]}...")

### Phase 2 — Transform: remap catalog/schema for target environment

This is the critical step. Every table identifier and every SQL reference inside `serialized_space` must be updated to point at the target environment's catalog and schema.

In a real CI/CD pipeline, this runs in your GitHub Actions / Azure DevOps / Jenkins step after checkout.

In [ ]:
# Simulate dev -> prod promotion
SOURCE_FQN = f"{CATALOG}.{SCHEMA}"
TARGET_FQN = f"{CATALOG}_prod.{SCHEMA}"  # example: workshop_demo_prod.genie_workshop_manufacturing

def remap_references(space_dict, source_fqn, target_fqn):
    # Deep string-replace of source_fqn -> target_fqn over the whole serialized_space JSON.
    raw = json.dumps(space_dict)
    remapped = raw.replace(source_fqn, target_fqn)
    return json.loads(remapped)

# Apply the transform
prod_space_def = remap_references(copy.deepcopy(space_def), SOURCE_FQN, TARGET_FQN)

# Verify: show table identifiers before and after
dev_tables = [t["identifier"] for t in space_def.get("data_sources", {}).get("tables", [])]
prod_tables = [t["identifier"] for t in prod_space_def.get("data_sources", {}).get("tables", [])]

print("DEV table identifiers:")
for t in dev_tables:
    print(f"  {t}")
print(f"\nPROD table identifiers (remapped {SOURCE_FQN} -> {TARGET_FQN}):")
for t in prod_tables:
    print(f"  {t}")

# In a real pipeline, you would commit this to Git:
# with open("genie_space_prod.json", "w") as f:
#     json.dump(prod_space_def, f, indent=2)
print("\nIn CI/CD: commit this JSON to version control before deploying.")

### Phase 3 — Deploy to target workspace

Create a new space (or update an existing one) in the target environment. In production, this runs as a **Lakeflow Job** with a **Service Principal** — not as a human user.

**For this workshop demo**, we deploy back into the same workspace with a `[CICD DEMO]` title prefix so you can see the result.

In [ ]:
import html  # safe link rendering if this cell runs standalone

# Phase 3: Deploy — create a new space from the transformed definition.
# NOTE: This uses the ORIGINAL space_def (not prod_space_def) because the prod catalog
# doesn't exist in this workshop. In a real pipeline, you'd use prod_space_def
# and target a different workspace host.

_wh_list = list(w.warehouses.list())
WAREHOUSE_ID = None
for _wh in _wh_list:
    _st = str(_wh.state).upper() if _wh.state else ""
    if _st in ("RUNNING", "STARTING") or getattr(_wh, "enable_serverless_compute", False):
        WAREHOUSE_ID = str(_wh.id)
        break
if not WAREHOUSE_ID and _wh_list:
    WAREHOUSE_ID = str(_wh_list[0].id)

if WAREHOUSE_ID:
    _src_title = exported.get("title") or exported.get("display_name") or "Manufacturing Analytics"
    deploy_body = {
        "title": f"[CICD DEMO] {_src_title}",
        "description": f"CI/CD deployed copy of the workshop space. Source space: {GENIE_SPACE_ID}",
        "warehouse_id": WAREHOUSE_ID,
        "serialized_space": json.dumps(space_def),  # use prod_space_def in real pipelines
    }

    deploy_resp = requests.post(
        f"{host_url}/api/2.0/genie/spaces",
        headers=headers,
        json=deploy_body,
    )

    if deploy_resp.status_code == 200:
        new_id = deploy_resp.json().get("space_id") or deploy_resp.json().get("id")
        new_url = genie_ui_room_url(host_url, new_id)
        print(f"Deployed! New space ID: {new_id}")
        print(f"URL: {new_url}")
        displayHTML(
            f'<p><a href=\"{html.escape(new_url, quote=True)}\" target=\"_blank\" rel=\"noopener\">'
            f"Open CI/CD deployed space</a></p>"
        )
    else:
        print(f"Deploy failed ({deploy_resp.status_code}): {deploy_resp.text[:500]}")
        print("This is expected if you lack CREATE permission. The pattern still works.")
else:
    print("No SQL warehouse found via API. Skipping deploy demo.")
    print("The export and transform steps above are the portable parts of the pipeline.")

### Updating an existing space (PATCH)

When promoting **changes** to a space that already exists in prod, use the round-trip update pattern. This preserves server-managed fields and existing conversation history.

In [ ]:
# Round-trip update pattern (documentation — does NOT run unless you uncomment):

# EXISTING_PROD_SPACE_ID = "<your-prod-space-id>"
#
# # 1. Fetch existing prod space
# prod_resp = requests.get(
#     f"{host_url}/api/2.0/genie/spaces/{EXISTING_PROD_SPACE_ID}?include_serialized_space=true",
#     headers=headers,
# )
# prod_resp.raise_for_status()
# prod_existing = json.loads(prod_resp.json()["serialized_space"])
#
# # 2. Apply targeted changes from dev export (e.g., update instructions only)
# prod_existing["instructions"]["text_instructions"] = prod_space_def["instructions"]["text_instructions"]
# prod_existing["instructions"]["example_question_sqls"] = prod_space_def["instructions"]["example_question_sqls"]
#
# # 3. PATCH back — preserves opaque IDs and server-managed fields
# patch_resp = requests.patch(
#     f"{host_url}/api/2.0/genie/spaces/{EXISTING_PROD_SPACE_ID}",
#     headers=headers,
#     json={"serialized_space": json.dumps(prod_existing)},
# )
# patch_resp.raise_for_status()
# print(f"Updated prod space: {EXISTING_PROD_SPACE_ID}")

print("Round-trip PATCH pattern shown above (commented out).")
print("Key: always fetch existing prod space first, apply targeted changes, then PATCH back.")
print("This preserves opaque IDs and server-managed fields.")

### CI/CD pipeline reference architecture

```
┌─────────────────────────────────────────────────────────────┐
│                     Git Repository                          │
│  genie_spaces/                                              │
│    manufacturing_quality.json   (serialized_space export)   │
│    transform.py                 (catalog/schema remapper)   │
│    deploy.py                    (POST/PATCH to target)      │
└────────────────────────┬────────────────────────────────────┘
                         │ PR merge triggers CI
                         ▼
┌─────────────────────────────────────────────────────────────┐
│               GitHub Actions / Azure DevOps                 │
│                                                             │
│  1. Checkout repo                                           │
│  2. Run transform.py (remap dev catalog → prod catalog)     │
│  3. Run deploy.py (POST or PATCH to prod workspace)         │
│  4. Run benchmarks against new space (notebook 06 via Job)  │
│  5. If benchmarks pass → done. If fail → alert + rollback.  │
└─────────────────────────────────────────────────────────────┘
```

### Field-built tools (open source)

| Tool | Author | What it does |
|------|--------|-------------|
| [SpaceOps CLI](https://github.com/charotAmine/databricks-spaceops) | Amine Charot | YAML-driven config, snapshot + deploy across environments |
| [genie-space-cicd](https://github.com/mohade09/genie-space-cicd) | Debadatta Mohapatra | Sample CI/CD repo for Genie spaces |
| [genie-migration-tool](https://github.com/reynoldspravindev/databricks-genie-migration-tool) | Reynolds Pravindev | CLI wrapper for CRUD APIs |

### Permissions tip

Set permissions on the **workspace folder** where you deploy the Genie space. The space inherits folder-level permissions — this is the cleanest access control pattern for CI/CD.

### What's coming

- **DABs native support** — PR `databricks/cli#4191` adds `genie_spaces` resource (DIRECT mode). Vote on [Aha DB-I-10709](https://databricks.aha.io/ideas/ideas/DB-I-10709).
- **Terraform** — on the roadmap after DABs GA.
- **Service Principal ownership** — not yet supported for Genie spaces; use SP for API calls but a user must currently own the space.